# TrendPulse — Part 1: Python Foundations & Data Structures

This notebook loads the daily *Trending YouTube Video Statistics* lists for three countries (US, GB, IN) and does all of the
cleaning, organising and labelling in **core Python** — loops, functions, conditions and the built-in data structures.

The only Pandas calls in the whole notebook are `pd.read_csv()` and `df.to_dict("records")`, both inside `load_country()`.

**Sections**
1. Setup and run header
2. Loading function and the category lookup
3. Type conversion and cleaning
4. Core data structures (list, tuple, set, dict, nested dict, slicing)
5. Variable scope
6. Loops over strings and sequences
7. Rule-based classification
8. Refactor

## 1. Setup

### Notebook discipline
* The notebook reads top to bottom: each section has a heading and a short explanation before its code.
* Configuration (paths, countries, thresholds) is declared once, here, and never redefined further down.
* No cell depends on a cell **below** it, and nothing was run out of order. Before committing I ran
  **Runtime → Restart and run all** (locally: *Kernel → Restart & Run All*) so the saved outputs come from one clean pass with no hidden state.
* There are no scratch or debugging cells; every cell is part of the analysis.
* Data is read only from this folder's `data/` subfolder. In Colab, clone the repository and `%cd` into
  `trendpulse/part1_python_foundations` first (or upload the `data/` folder next to the notebook).

In [1]:
import os
from datetime import datetime, date

import pandas as pd   # used ONLY for pd.read_csv() and df.to_dict("records") inside load_country()

# ---- configuration -------------------------------------------------------
DATA_DIR = "data"                                    # str
COUNTRIES = ["US", "GB", "IN"]                       # list[str] - order is the load order
COUNTRY_FILES = {c: os.path.join(DATA_DIR, f"{c}videos.csv") for c in COUNTRIES}   # dict[str, str]
CATEGORIES_FILE = os.path.join(DATA_DIR, "categories.csv")

REQUIRED_COLUMNS = ("video_id", "trending_date", "title", "channel_title", "category_id",
                    "publish_time", "views", "likes", "dislikes", "comment_count",
                    "comments_disabled", "ratings_disabled", "video_error_or_removed")   # tuple: fixed

INT_FIELDS = ("views", "likes", "dislikes", "comment_count")
BOOL_FIELDS = ("comments_disabled", "ratings_disabled", "video_error_or_removed")

# classification thresholds (justified in section 7)
VIRAL_VIEWS = 2_000_000        # int  - roughly the top 10% of trending records
STEADY_VIEWS = 100_000         # int  - below this a trending record is a small one
VIRAL_MIN_RATIO = 0.90         # float - a viral hit must also be well liked
DIVISIVE_RATIO = 0.80          # float - fewer than 80% likes among votes = divisive

SCAN_VIEW_THRESHOLD = 10_000_000   # int - used by the while-loop demo in section 6
TOP_N_CHANNELS = 15                # int

RUN_STARTED = datetime.now()       # datetime

print("=" * 62)
print(f"{'TrendPulse - Part 1 run':^62}")
print("=" * 62)
print(f"Run started : {RUN_STARTED:%Y-%m-%d %H:%M:%S}")
print(f"Countries   : {', '.join(COUNTRIES)}  ({len(COUNTRIES)} files)")
for c, path in COUNTRY_FILES.items():
    print(f"   {c:<3} -> {path}")
print(f"Lookup      : {CATEGORIES_FILE}")
print(f"Thresholds  : viral >= {VIRAL_VIEWS:,} views, steady >= {STEADY_VIEWS:,} views, "
      f"divisive ratio < {DIVISIVE_RATIO:.0%}")
print("=" * 62)

                   TrendPulse - Part 1 run                    
Run started : 2026-11-08 17:21:10
Countries   : US, GB, IN  (3 files)
   US  -> data/USvideos.csv
   GB  -> data/GBvideos.csv
   IN  -> data/INvideos.csv
Lookup      : data/categories.csv
Thresholds  : viral >= 2,000,000 views, steady >= 100,000 views, divisive ratio < 80%


## 2. Loading function

`load_country()` takes the file path and the country code, has a **default argument** `max_rows=None` (load everything unless told
otherwise) and **returns** a list of dictionaries — one dictionary per CSV row.

I read every column as text (`dtype=str, keep_default_na=False`). That way Pandas does *no* type guessing, blanks arrive as the empty
string `""`, and **all** type conversion happens in core Python in section 3, where I can count and justify every failure.

In [2]:
def load_country(path, country, max_rows=None):
    """Read one country's trending CSV and return its rows as a list of dicts (all values are strings)."""
    df = pd.read_csv(path, dtype=str, keep_default_na=False, nrows=max_rows)
    missing = [col for col in REQUIRED_COLUMNS if col not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing required columns: {missing}")
    rows = df.to_dict("records")
    for row in rows:
        row["country"] = country
    return rows


raw_by_country = {}
for country in COUNTRIES:
    raw_by_country[country] = load_country(COUNTRY_FILES[country], country)
    print(f"{country}: loaded {len(raw_by_country[country]):>6,} records")

raw_records = []
for country in COUNTRIES:
    raw_records.extend(raw_by_country[country])
print(f"Total: {len(raw_records):,} records from {len(raw_by_country)} countries")
print("Columns:", list(raw_records[0].keys()))

US: loaded 16,080 records


GB: loaded 15,075 records


IN: loaded 15,075 records
Total: 46,230 records from 3 countries
Columns: ['video_id', 'trending_date', 'title', 'channel_title', 'category_id', 'publish_time', 'tags', 'views', 'likes', 'dislikes', 'comment_count', 'thumbnail_link', 'comments_disabled', 'ratings_disabled', 'video_error_or_removed', 'description', 'country']


A quick check that the default argument works — `max_rows=3` loads only three rows:

In [3]:
sample = load_country(COUNTRY_FILES["US"], "US", max_rows=3)
print(len(sample), "rows ->", [r["video_id"] for r in sample])

3 rows -> ['g_N-w8LeAi8', 'Ugyql2NlPAd', '8fE19jkxJ9n']


### Category lookup
`category_names` maps each numeric `category_id` to its name. A dictionary gives O(1) lookup by key, which is exactly how it is used
(once per record).

In [4]:
category_rows = pd.read_csv(CATEGORIES_FILE, dtype=str, keep_default_na=False).to_dict("records")
category_names = {}
for row in category_rows:
    category_names[int(row["category_id"])] = row["category_name"].strip()

print(f"{len(category_names)} categories")
for cid, name in category_names.items():
    print(f"  {cid:>3} -> {name}")

17 categories
    1 -> Film & Animation
    2 -> Autos & Vehicles
   10 -> Music
   15 -> Pets & Animals
   17 -> Sports
   19 -> Travel & Events
   20 -> Gaming
   22 -> People & Blogs
   23 -> Comedy
   24 -> Entertainment
   25 -> News & Politics
   26 -> Howto & Style
   27 -> Education
   28 -> Science & Technology
   29 -> Nonprofits & Activism
   30 -> Movies
   43 -> Shows


## 3. Type conversion and cleaning (core Python)

Every value is still a string. I convert them with small reusable functions, each wrapped in `try/except` so one bad value can never crash
the loop, and every failure is counted per field.

### Cleaning rules and why

| Field | Conversion | Blank / bad value | Why |
|---|---|---|---|
| `views` | `int` | **drop the record** | Views are the core metric of a trending record; a row without them cannot be ranked or classified, and inventing a number would distort every average. |
| `likes`, `dislikes`, `comment_count` | `int` (via `float`, because the file stores likes as `2155.0`) | **keep the record, store `None`** | One missing count should not throw away the views and dates that are still valid. `None` (not `0`) keeps "unknown" separate from a genuine zero, and the classifier treats `None` explicitly. |
| flag columns | `"True"`/`"False"` → `bool` | substitute `False`, count the failure | `False` is the platform default (comments/ratings on, video live). |
| `trending_date` | `yy.dd.mm` → `datetime.date` | **drop the record** | A trending record without its trending day cannot be placed in time, deduplicated or counted towards a lifespan. |
| `publish_time` | ISO `2017-11-13T17:13:01.000Z` → `datetime` | keep the record, store `None` | Publish time is only needed for time-to-trend, not for the record itself. |
| `category_id` | `int` | keep, `None` | the category name becomes `"Unknown"`. |

Two further rules:
* **Removed / errored videos** (`video_error_or_removed == True`) are dropped: their statistics are frozen or invalid.
* **Duplicates**: the same `(country, video_id, trending_date)` seen twice is the same observation recorded twice; only the first is kept.
  The key is a **tuple**, so it can live in a `set`.

**Negative counts** are treated as bad values (they cannot happen on YouTube).

In [5]:
def to_int(value):
    """'2155.0' / '2155' -> 2155. Returns None for blanks; raises ValueError for anything else invalid."""
    text = value.strip()
    if text == "":
        return None
    number = float(text)                 # handles '2155.0'; raises ValueError for 'abc'
    if number < 0 or not number.is_integer():
        raise ValueError(f"not a valid count: {value!r}")
    return int(number)


def to_bool(value):
    text = value.strip().lower()
    if text in ("true", "1", "yes"):
        return True
    if text in ("false", "0", "no"):
        return False
    raise ValueError(f"not a boolean: {value!r}")


def parse_trending_date(value):
    """The dataset stores trending_date as yy.dd.mm, e.g. '17.14.11' = 14 Nov 2017."""
    yy, dd, mm = value.strip().split(".")          # raises ValueError if not three parts
    return date(2000 + int(yy), int(mm), int(dd))  # raises ValueError for an impossible date


def parse_publish_time(value):
    return datetime.strptime(value.strip(), "%Y-%m-%dT%H:%M:%S.%fZ")


# quick checks of the converters on tricky inputs
print(to_int("2155.0"), to_int(""), to_bool("False"), parse_trending_date("17.14.11"),
      parse_publish_time("2017-11-13T17:13:01.000Z"))
for bad in ("abc", "-5", "3.7"):
    try:
        to_int(bad)
    except ValueError as err:
        print("rejected:", err)

2155 None False 2017-11-14 2017-11-13 17:13:01
rejected: could not convert string to float: 'abc'
rejected: not a valid count: '-5'
rejected: not a valid count: '3.7'


In [6]:
parse_failures = {f: 0 for f in INT_FIELDS + BOOL_FIELDS + ("trending_date", "publish_time", "category_id")}
blank_values = {f: 0 for f in INT_FIELDS}
dropped = {"missing/bad views": 0, "bad trending_date": 0, "removed/errored video": 0, "duplicate": 0}

clean_records = []
seen_keys = set()

for raw in raw_records:
    rec = {"country": raw["country"], "video_id": raw["video_id"].strip(),
           "title": raw["title"], "channel_title": raw["channel_title"].strip(), "tags": raw["tags"]}

    # integers
    for field in INT_FIELDS:
        try:
            rec[field] = to_int(raw[field])
            if rec[field] is None:
                blank_values[field] += 1
        except ValueError:
            parse_failures[field] += 1
            rec[field] = None

    # booleans
    for field in BOOL_FIELDS:
        try:
            rec[field] = to_bool(raw[field])
        except ValueError:
            parse_failures[field] += 1
            rec[field] = False

    # dates
    try:
        rec["trending_date"] = parse_trending_date(raw["trending_date"])
    except ValueError:
        parse_failures["trending_date"] += 1
        rec["trending_date"] = None
    try:
        rec["publish_time"] = parse_publish_time(raw["publish_time"])
    except ValueError:
        parse_failures["publish_time"] += 1
        rec["publish_time"] = None

    # category
    try:
        rec["category_id"] = int(raw["category_id"])
    except ValueError:
        parse_failures["category_id"] += 1
        rec["category_id"] = None
    rec["category_name"] = category_names.get(rec["category_id"], "Unknown")

    # drop rules
    if rec["views"] is None:
        dropped["missing/bad views"] += 1
        continue
    if rec["trending_date"] is None:
        dropped["bad trending_date"] += 1
        continue
    if rec["video_error_or_removed"]:
        dropped["removed/errored video"] += 1
        continue
    key = (rec["country"], rec["video_id"], rec["trending_date"])
    if key in seen_keys:
        dropped["duplicate"] += 1
        continue
    seen_keys.add(key)
    clean_records.append(rec)

print(f"{'field':<24}{'parse failures':>15}{'blank':>8}")
for field, n_fail in parse_failures.items():
    print(f"{field:<24}{n_fail:>15}{blank_values.get(field, ''):>8}")
print()
for reason, n in dropped.items():
    print(f"dropped ({reason}): {n}")
print(f"\nClean records: {len(clean_records):,} of {len(raw_records):,} "
      f"({len(clean_records) / len(raw_records):.1%} kept)")

field                    parse failures   blank
views                                 0       0
likes                                 0      12
dislikes                              0       0
comment_count                         0      36
comments_disabled                     0        
ratings_disabled                      0        
video_error_or_removed                0        
trending_date                         0        
publish_time                          0        
category_id                           0        

dropped (missing/bad views): 0
dropped (bad trending_date): 0
dropped (removed/errored video): 92
dropped (duplicate): 230

Clean records: 45,908 of 46,230 (99.3% kept)


**Result.** No value in this copy was *unparseable* (all failure counts are 0), but there were genuine blanks: missing `likes` and
`comment_count` values (kept as `None`, see the table above), removed/errored videos and exact duplicate rows (both dropped).
`trending_date` really is `yy.dd.mm` in this copy — the converter turned `17.14.11` into 14 November 2017, and no date failed. The
try/except blocks are still needed: they are what makes the loop safe if a future file contains `"abc"` in `views`.

In [7]:
first = clean_records[0]
for k, v in first.items():
    print(f"{k:<24} {type(v).__name__:<9} {v!r}"[:110])

country                  str       'US'
video_id                 str       'g_N-w8LeAi8'
title                    str       'Honest Football Prank LEAKED'
channel_title            str       'QuietSpeech10'
tags                     str       'Monsoon|Robots'
views                    int       48555
likes                    int       2189
dislikes                 int       180
comment_count            int       136
comments_disabled        bool      False
ratings_disabled         bool      False
video_error_or_removed   bool      False
trending_date            date      datetime.date(2017, 11, 14)
publish_time             datetime  datetime.datetime(2017, 11, 14, 7, 4, 28)
category_id              int       22
category_name            str       'People & Blogs'


## 4. Core data structures

### 4.1 List of dictionaries and tuples
* `clean_records` (already built) is a **list of dictionaries**: a list keeps the original row order and allows any number of records;
  each record is a dictionary because its fields are accessed by *name* (`rec["views"]`) and new fields (like `label`) can be added later.
  Both are **mutable**.
* For the four key fields I build **tuples** `(video_id, trending_date, views, likes)`. A trending record is a fixed historical observation:
  on that date the video had that many views. A tuple is **immutable**, so it cannot be changed by accident, it is lighter than a dict, and it
  is **hashable**, so it can be a set member or a dictionary key (I already used a tuple key for deduplication in section 3).

In [8]:
key_facts = [(r["video_id"], r["trending_date"], r["views"], r["likes"]) for r in clean_records]
print(len(key_facts), "tuples; first three:")
for t in key_facts[:3]:
    print("  ", t)

try:
    key_facts[0][2] = 0
except TypeError as err:
    print("Tuples cannot be modified:", err)

45908 tuples; first three:
   ('g_N-w8LeAi8', datetime.date(2017, 11, 14), 48555, 2189)
   ('Ugyql2NlPAd', datetime.date(2017, 11, 14), 198385, 1159)
   ('8fE19jkxJ9n', datetime.date(2017, 11, 14), 231656, 8742)
Tuples cannot be modified: 'tuple' object does not support item assignment


### 4.2 Sets — which videos trended in more than one country?
A **set** stores each video ID once and makes membership, intersection (`&`) and difference (`-`) fast and readable.
Sets are mutable (you can add to them), but their *elements* must be immutable (strings here).

In [9]:
ids_by_country = {}
for country in COUNTRIES:
    ids_by_country[country] = set()
for r in clean_records:
    ids_by_country[r["country"]].add(r["video_id"])

for country, ids in ids_by_country.items():
    print(f"{country}: {len(ids):,} unique video IDs")

us, gb, india = ids_by_country["US"], ids_by_country["GB"], ids_by_country["IN"]
print()
print(f"US & GB          : {len(us & gb):>5}")
print(f"US & IN          : {len(us & india):>5}")
print(f"GB & IN          : {len(gb & india):>5}")
print(f"all three        : {len(us & gb & india):>5}")

in_multiple = (us & gb) | (us & india) | (gb & india)
print(f"in 2+ countries  : {len(in_multiple):>5}")
print()
print(f"only US          : {len(us - gb - india):>5}")
print(f"only GB          : {len(gb - us - india):>5}")
print(f"only IN          : {len(india - us - gb):>5}")
print("examples of videos trending in all three:", sorted(us & gb & india)[:5])

US: 5,064 unique video IDs
GB: 4,112 unique video IDs
IN: 5,871 unique video IDs

US & GB          :   134
US & IN          :   119
GB & IN          :   126
all three        :    20
in 2+ countries  :   339

only US          :  4831
only GB          :  3872
only IN          :  5646
examples of videos trending in all three: ['0B3bCL2UO2y', '2i0BT5fDj1C', '5_vLwTVq3s-', '8HDRONbJrZG', 'A-U6P8eTT7f']


### 4.3 Dictionary counter + slicing — the busiest channels
A **dictionary** maps each channel name to its count of trending appearances; looking up and incrementing a key is O(1).
To get the top 15 I sort the `(channel, count)` pairs by count and **slice** the first 15 with `[:TOP_N_CHANNELS]`.

In [10]:
channel_counts = {}
for r in clean_records:
    channel = r["channel_title"]
    if channel in channel_counts:
        channel_counts[channel] += 1
    else:
        channel_counts[channel] = 1

ranked_channels = sorted(channel_counts.items(), key=lambda pair: pair[1], reverse=True)
print(f"{len(channel_counts)} distinct channels. Top {TOP_N_CHANNELS} by trending appearances:")
for rank, (channel, n) in enumerate(ranked_channels[:TOP_N_CHANNELS], start=1):
    print(f"{rank:>3}. {channel:<26} {n:>5}")

438 distinct channels. Top 15 by trending appearances:
  1. QuietChallenge75            2448
  2. OfficialReview41            1575
  3. UltimateUnboxing65          1423
  4. InsaneReaction46            1223
  5. Brand NewChallenge45        1169
  6. CrazyTrailer91              1010
  7. FinalTrailer17               902
  8. UltimateSpeech20             766
  9. WildHighlights7              682
 10. Brand NewSecret51            679
 11. SurprisingHighlights41       648
 12. FinalBreakdown11             641
 13. CrazyRecipe67                600
 14. QuietInterview9              564
 15. UltimateVlog96               561


### 4.4 Nested dictionary — country → category → number of records
A nested dictionary mirrors the two-level question ("for this country, how many records per category?"):
`summary["GB"]["Music"]` reads exactly like the question. Both levels are mutable dictionaries.

In [11]:
country_category = {}
for r in clean_records:
    country, cat = r["country"], r["category_name"]
    if country not in country_category:
        country_category[country] = {}
    if cat in country_category[country]:
        country_category[country][cat] += 1
    else:
        country_category[country][cat] = 1

for country in COUNTRIES:
    cats = sorted(country_category[country].items(), key=lambda pair: pair[1], reverse=True)
    total = sum(country_category[country].values())
    print(f"{country} ({total:,} records) - top 5 categories:")
    for cat, n in cats[:5]:
        print(f"    {cat:<22} {n:>5}  ({n / total:.1%})")

US (15,968 records) - top 5 categories:
    Entertainment           4342  (27.2%)
    Music                   3029  (19.0%)
    People & Blogs          1416  (8.9%)
    Comedy                  1358  (8.5%)
    Film & Animation         816  (5.1%)
GB (14,970 records) - top 5 categories:
    Music                   4978  (33.3%)
    Entertainment           2821  (18.8%)
    People & Blogs          1289  (8.6%)
    Comedy                   935  (6.2%)
    Film & Animation         708  (4.7%)
IN (14,970 records) - top 5 categories:
    Entertainment           5081  (33.9%)
    Music                   2279  (15.2%)
    News & Politics         1136  (7.6%)
    People & Blogs          1092  (7.3%)
    Comedy                   958  (6.4%)


### 4.5 Summary of the structures

| Structure | Used for | Why this structure | Mutable? |
|---|---|---|---|
| `list` of `dict` | `clean_records` | ordered, any length; each field accessed by name; new fields can be added | list: mutable, dicts: mutable |
| `tuple` | `key_facts`, dedup keys `(country, video_id, trending_date)` | fixed facts that must not change; hashable so they can go into a set | **immutable** |
| `set` | unique video IDs per country | automatic uniqueness; `&` and `-` answer "in both" / "only here" directly | mutable (elements immutable) |
| `dict` | `channel_counts`, `category_names` | O(1) lookup/increment by key | mutable |
| nested `dict` | `country_category` | two-level grouping maps directly to the question | mutable |
| slicing | `ranked_channels[:15]` | take the first N of an already sorted list | (returns a new list) |

## 5. Variable scope

`trend_label` is defined **outside** any function (global scope), so a function can read it. `local_note` is defined **inside**
`describe_scope()`, so it exists only while that function runs: trying to read it from outside raises `NameError`.
Assigning to a name inside a function creates a *new local* variable — it does not change the global one unless `global` is declared.

In [12]:
trend_label = "global: visible everywhere in this notebook"

def describe_scope():
    local_note = "local: only exists inside describe_scope()"
    print("inside  -> can read the global :", trend_label)
    print("inside  -> can read the local  :", local_note)

def shadow():
    trend_label = "a NEW local variable with the same name"
    print("inside shadow() ->", trend_label)

describe_scope()
try:
    print(local_note)
except NameError as err:
    print("outside -> NameError:", err)

shadow()
print("outside after shadow() -> global unchanged:", trend_label)

inside  -> can read the global : global: visible everywhere in this notebook
inside  -> can read the local  : local: only exists inside describe_scope()
outside -> NameError: name 'local_note' is not defined
inside shadow() -> a NEW local variable with the same name
outside after shadow() -> global unchanged: global: visible everywhere in this notebook


## 6. Loops over strings and sequences

`title_features(title)` walks through a title in three ways:
* a `for` loop over every **character** counts `!` and `?`;
* `range(len(words))` with **index access** walks the words and counts fully UPPERCASE words; words with no letters
  (like `-` or `(Live)` punctuation) are skipped with `continue`, and single letters such as "I" do not count as shouting;
* the length in words is the number of words.

In [13]:
def title_features(title):
    marks = 0
    for ch in title:                     # loop over characters
        if ch == "!" or ch == "?":
            marks += 1

    words = title.split()
    upper_words = 0
    for i in range(len(words)):          # range() with index access
        word = words[i]
        letters = ""
        for ch in word:
            if ch.isalpha():
                letters += ch
        if len(letters) < 2:             # '-', '&', 'I' ... not shouting
            continue
        if letters.isupper():
            upper_words += 1

    return {"upper_words": upper_words, "marks": marks, "n_words": len(words)}


for t in ["Wild Winter Prank", "Perfect Pasta Unboxing?!", "EPIC CHESS REVIEW LIVE!!", "Kairo - Space (Official Video)"]:
    print(f"{t:<34} {title_features(t)}")

Wild Winter Prank                  {'upper_words': 0, 'marks': 0, 'n_words': 3}
Perfect Pasta Unboxing?!           {'upper_words': 0, 'marks': 2, 'n_words': 3}
EPIC CHESS REVIEW LIVE!!           {'upper_words': 4, 'marks': 2, 'n_words': 4}
Kairo - Space (Official Video)     {'upper_words': 0, 'marks': 0, 'n_words': 5}


In [14]:
shouty = 0
with_marks = 0
total_words = 0
for r in clean_records:
    f = title_features(r["title"])
    r["title_features"] = f
    total_words += f["n_words"]
    if f["upper_words"] > 0:
        shouty += 1
    if f["marks"] > 0:
        with_marks += 1

n = len(clean_records)
print(f"records with at least one ALL-CAPS word : {shouty:,} ({shouty / n:.1%})")
print(f"records with ! or ?                     : {with_marks:,} ({with_marks / n:.1%})")
print(f"average title length                    : {total_words / n:.1f} words")

records with at least one ALL-CAPS word : 14,016 (30.5%)
records with ! or ?                     : 9,000 (19.6%)
average title length                    : 3.9 words


### `while` loop with `break` and `continue`
Scan the records in file order until the **first** one above `SCAN_VIEW_THRESHOLD` views. The loop ends either at `break`
(found one) or when the index reaches the end of the list (`i < len(clean_records)`), so it always terminates. Records with unknown
likes are skipped with `continue` (I want a fully-known record); the index is increased **before** `continue`, otherwise the loop would
never move on.

In [15]:
i = 0
found = None
skipped = 0
while i < len(clean_records):
    r = clean_records[i]
    i += 1
    if r["likes"] is None:
        skipped += 1
        continue
    if r["views"] > SCAN_VIEW_THRESHOLD:
        found = r
        break

if found is None:
    print("No record above the threshold.")
else:
    print(f"Scanned {i:,} records (skipped {skipped} with unknown likes).")
    print(f"First record above {SCAN_VIEW_THRESHOLD:,} views: {found['title']!r} "
          f"[{found['country']}, {found['trending_date']}] with {found['views']:,} views")

Scanned 1,026 records (skipped 0 with unknown likes).
First record above 10,000,000 views: 'Blue Harbor - Cars (Live)' [US, 2017-11-19] with 10,100,256 views


## 7. Rule-based classification

### Thresholds
* `VIRAL_VIEWS = 2,000,000` — about the top 10% of trending records in this data, so "viral" means genuinely exceptional.
* `STEADY_VIEWS = 100,000` — below this, even a trending record is small.
* `DIVISIVE_RATIO = 0.80` — normally ~95% of votes on a trending video are likes; under 80% means a large share of the audience disliked it.
* `VIRAL_MIN_RATIO = 0.90` — a viral hit must also be well liked.
* Like ratio = `likes / (likes + dislikes)`.

### Rules (checked in this order)
1. views missing or `0` → **Low Engagement** (nothing to judge; prevents any later division by views).
2. ratings disabled, or likes/dislikes missing → **Unrated** (a like ratio cannot be computed).
3. likes + dislikes = 0 → **Unrated** (nobody voted; the ratio would be 0/0).
4. ratio < 0.80 → **Divisive** (regardless of views).
5. views ≥ 2,000,000:
   * ratio ≥ 0.90 **and** comments enabled → **Viral Hit**
   * otherwise → **Steady Performer** (big, but not clearly loved, or with comments switched off we cannot see the conversation a hit creates)
6. views ≥ 100,000 → **Steady Performer**
7. else → **Low Engagement**

In [16]:
def classify_video(record):
    views = record.get("views")
    likes = record.get("likes")
    dislikes = record.get("dislikes")

    if views is None or views == 0:
        return "Low Engagement"
    elif record.get("ratings_disabled") or likes is None or dislikes is None:
        return "Unrated"
    else:
        votes = likes + dislikes
        if votes == 0:
            return "Unrated"
        ratio = likes / votes                    # safe: votes > 0 here
        if ratio < DIVISIVE_RATIO:
            return "Divisive"
        elif views >= VIRAL_VIEWS:
            if ratio >= VIRAL_MIN_RATIO and not record.get("comments_disabled"):
                return "Viral Hit"
            else:
                return "Steady Performer"
        elif views >= STEADY_VIEWS:
            return "Steady Performer"
        else:
            return "Low Engagement"


edge_cases = [
    {"views": 0, "likes": 0, "dislikes": 0},
    {"views": 5_000, "likes": 0, "dislikes": 0},
    {"views": 3_000_000, "likes": 0, "dislikes": 0, "ratings_disabled": True},
    {"views": 3_000_000, "likes": None, "dislikes": 10},
    {"views": None, "likes": 10, "dislikes": 1},
    {"views": 3_000_000, "likes": 95_000, "dislikes": 2_000, "comments_disabled": True},
]
for case in edge_cases:
    print(f"{str(case):<90} -> {classify_video(case)}")

{'views': 0, 'likes': 0, 'dislikes': 0}                                                    -> Low Engagement
{'views': 5000, 'likes': 0, 'dislikes': 0}                                                 -> Unrated
{'views': 3000000, 'likes': 0, 'dislikes': 0, 'ratings_disabled': True}                    -> Unrated
{'views': 3000000, 'likes': None, 'dislikes': 10}                                          -> Unrated
{'views': None, 'likes': 10, 'dislikes': 1}                                                -> Low Engagement
{'views': 3000000, 'likes': 95000, 'dislikes': 2000, 'comments_disabled': True}            -> Steady Performer


### Hand-trace of one real record
Let me trace the first US record in the file: *"Honest Football Prank LEAKED"* — views = 48,555, likes = 2,189, dislikes = 180,
ratings and comments enabled.

1. `views` is 48,555 — not `None` and not 0 → skip the first branch.
2. ratings are enabled and both likes and dislikes are known → skip the "Unrated" branch, enter `else`.
3. `votes = 2,189 + 180 = 2,369`, which is not 0.
4. `ratio = 2,189 / 2,369 = 0.924` — not below 0.80, so not Divisive.
5. `views >= 2,000,000`? No.
6. `views >= 100,000`? No (48,555).
7. → **Low Engagement**: a well-liked but small video. The cell below confirms it.

In [17]:
traced = clean_records[0]
votes = traced["likes"] + traced["dislikes"]
print(traced["title"], "|", traced["views"], traced["likes"], traced["dislikes"],
      f"| ratio = {traced['likes'] / votes:.3f} ->", classify_video(traced))

Honest Football Prank LEAKED | 48555 2189 180 | ratio = 0.924 -> Low Engagement


In [18]:
label_counts = {}
for r in clean_records:
    label = classify_video(r)
    r["label"] = label
    if label in label_counts:
        label_counts[label] += 1
    else:
        label_counts[label] = 1

print(f"{'label':<18}{'records':>9}{'share':>8}")
for label, n in sorted(label_counts.items(), key=lambda pair: pair[1], reverse=True):
    print(f"{label:<18}{n:>9,}{n / len(clean_records):>8.1%}")
print(f"{'total':<18}{sum(label_counts.values()):>9,}")

label               records   share
Steady Performer     31,191   67.9%
Low Engagement        8,005   17.4%
Viral Hit             4,236    9.2%
Divisive              1,925    4.2%
Unrated                 551    1.2%
total                45,908


## 8. Refactor

The block *"if the key is in the dictionary add 1, otherwise set it to 1"* appears three times above: channel counts (4.3), the
nested country → category counts (4.4) and the label counts (7). I refactor it into one function, `count_by(items, key_func)`, and check
that it reproduces all three results exactly.

In [19]:
def count_by(items, key_func):
    """Count items by the value key_func(item) returns. Returns a dict {key: count}."""
    counts = {}
    for item in items:
        key = key_func(item)
        counts[key] = counts.get(key, 0) + 1
    return counts


channel_counts_v2 = count_by(clean_records, lambda r: r["channel_title"])
label_counts_v2 = count_by(clean_records, lambda r: r["label"])
country_category_v2 = {}
for country in COUNTRIES:
    rows = [r for r in clean_records if r["country"] == country]
    country_category_v2[country] = count_by(rows, lambda r: r["category_name"])

print("channel counts identical :", channel_counts_v2 == channel_counts)
print("label counts identical   :", label_counts_v2 == label_counts)
print("nested summary identical :", country_category_v2 == country_category)

channel counts identical : True
label counts identical   : True
nested summary identical : True


**What I gained:** the counting logic now lives in one tested place, so a fix or change (for example, ignoring `None` keys) is made
once instead of three times. Each new count is a single readable line — `count_by(records, lambda r: r["label"])` — that says *what* is
counted rather than repeating *how*.